# V1 Static ASL Classifier — Kaggle GPU Training

This notebook trains the V1 static ASL classifier using MobileNetV3-Small and the project's shared modules. It does **not** use the local ZIP archive. Attach the Kaggle **ASL Alphabet** dataset as a notebook input; Kaggle provides its extracted files under `/kaggle/input/`.

Before running: enable a GPU accelerator in Kaggle. To reuse the latest project code, either attach a Kaggle Dataset containing this repository or enable Internet access so the setup cell can clone the GitHub repository. The resulting best `.pth` checkpoint is written to `/kaggle/working/` and appears in the notebook's **Output** pane for download after the run completes.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

# Update these only if your Kaggle input mount names differ.
ASL_DATASET_DIR = Path('/kaggle/input/asl-alphabet')
PROJECT_SOURCE_INPUT = Path('/kaggle/input/dynamic-sign-language-translator')
REPOSITORY_URL = 'https://github.com/aadit1007/dynamic-sign-language-translator.git'
PROJECT_ROOT = Path('/kaggle/working/dynamic-sign-language-translator')

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
if PROJECT_SOURCE_INPUT.exists():
    # Preferred offline option: attach a Kaggle Dataset containing the project source.
    shutil.copytree(PROJECT_SOURCE_INPUT, PROJECT_ROOT)
else:
    # Requires Kaggle Internet to be enabled and the required source to be on GitHub.
    subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)

ML_ROOT = PROJECT_ROOT / 'ml'
if not (ML_ROOT / 'src' / 'config.py').is_file():
    raise FileNotFoundError('Project source must include ml/src/config.py.')
if not ASL_DATASET_DIR.is_dir():
    raise FileNotFoundError(f'Attach the ASL Alphabet Kaggle dataset at {ASL_DATASET_DIR}.')
sys.path.insert(0, str(ML_ROOT))
OUTPUT_DIR = Path('/kaggle/working')
CHECKPOINT_PATH = OUTPUT_DIR / 'best_mobilenetv3_small_v1.pth'


In [ ]:
import random
import time
from collections import Counter

import torch
import torchvision
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
from PIL import Image

from src.config import (
    ASL_CLASSES,
    DATASET_CLASS_TO_V1,
    DEFAULT_BATCH_SIZE,
    DEFAULT_EPOCHS,
    DEFAULT_LEARNING_RATE,
    DEFAULT_WEIGHT_DECAY,
    EXCLUDED_DATASET_CLASSES,
    RANDOM_SEED,
)
from src.evaluate import evaluate_model
from src.model import create_model, parameter_count
from src.train import checkpoint_payload, train_one_epoch
from src.transforms import evaluation_transform, training_transform

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')
print(f'PyTorch: {torch.__version__}')
print(f'torchvision: {torchvision.__version__}')
print(f'V1 classes ({len(ASL_CLASSES)}): {list(ASL_CLASSES)}')
print(f'Class mapping: {DATASET_CLASS_TO_V1}')
print(f'Excluded source classes: {sorted(EXCLUDED_DATASET_CLASSES)}')


## Reproducible split

Only the source training folder is used. The dataset's small separate test folder is intentionally ignored. Each selected source folder has 3,000 images and is shuffled using seed 42, then assigned 2,400 / 300 / 300 examples to train / validation / test. `J` and `Z` are never added to a split.

In [ ]:
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png'}

def find_training_root(dataset_dir: Path) -> Path:
    candidates = sorted(
        path for path in dataset_dir.rglob('asl_alphabet_train')
        if path.is_dir() and (path / 'A').is_dir()
    )
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one training root containing A/, found: {candidates}')
    return candidates[0]

TRAIN_ROOT = find_training_root(ASL_DATASET_DIR)

def build_records(training_root: Path, seed: int = RANDOM_SEED):
    records = {split: [] for split in ('train', 'validation', 'test')}
    rng = random.Random(seed)
    for label in ASL_CLASSES:
        source_name = next(name for name, mapped in DATASET_CLASS_TO_V1.items() if mapped == label)
        paths = sorted(path for path in (training_root / source_name).iterdir()
                       if path.suffix.lower() in IMAGE_SUFFIXES)
        if len(paths) != 3000:
            raise ValueError(f'{source_name} has {len(paths)} images; expected 3000.')
        rng.shuffle(paths)
        records['train'].extend((path, label) for path in paths[:2400])
        records['validation'].extend((path, label) for path in paths[2400:2700])
        records['test'].extend((path, label) for path in paths[2700:])
    return records

RECORDS = build_records(TRAIN_ROOT)
for split, records in RECORDS.items():
    counts = Counter(label for _, label in records)
    print(f'{split}: {len(records)} images; class counts={set(counts.values())}; classes={len(counts)}')
assert all('J' != label and 'Z' != label for records in RECORDS.values() for _, label in records)
assert all(set(label for _, label in records) == set(ASL_CLASSES) for records in RECORDS.values())


In [ ]:
class KaggleASLDataset(Dataset):
    """Thin filesystem adapter; model, transforms, training, and metrics stay shared."""
    def __init__(self, records, transform):
        self.records = records
        self.transform = transform
        self.class_to_index = {label: index for index, label in enumerate(ASL_CLASSES)}

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        image_path, label = self.records[index]
        with Image.open(image_path) as image:
            image = image.convert('RGB')
        return self.transform(image), self.class_to_index[label]

# Adjust these before the training cell if GPU memory requires it.
BATCH_SIZE = DEFAULT_BATCH_SIZE
EPOCHS = DEFAULT_EPOCHS
NUM_WORKERS = 2
LEARNING_RATE = DEFAULT_LEARNING_RATE
WEIGHT_DECAY = DEFAULT_WEIGHT_DECAY

datasets = {
    'train': KaggleASLDataset(RECORDS['train'], training_transform()),
    'validation': KaggleASLDataset(RECORDS['validation'], evaluation_transform()),
    'test': KaggleASLDataset(RECORDS['test'], evaluation_transform()),
}
loaders = {
    split: DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=split == 'train',
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
    )
    for split, dataset in datasets.items()
}
model = create_model(pretrained=True).to(DEVICE)
print(f'Model parameters: {parameter_count(model):,}')
print(f'Classifier outputs: {model.classifier[-1].out_features}')


## Training

Run this cell only when ready. The first run may download ImageNet weights. The checkpoint is overwritten only when validation accuracy improves.

In [ ]:
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
training_config = {
    'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY, 'seed': RANDOM_SEED, 'pretrained': True,
}
history = []
best_validation_accuracy = -1.0

for epoch in range(1, EPOCHS + 1):
    started = time.perf_counter()
    train_metrics = train_one_epoch(model, loaders['train'], optimizer, DEVICE)
    validation_metrics = evaluate_model(model, loaders['validation'], DEVICE)
    scheduler.step()
    epoch_seconds = time.perf_counter() - started
    row = {
        'epoch': epoch, 'train_loss': train_metrics['loss'], 'train_accuracy': train_metrics['accuracy'],
        'validation_loss': validation_metrics['loss'],
        'validation_accuracy': validation_metrics['accuracy'], 'epoch_seconds': epoch_seconds,
    }
    history.append(row)
    print(row)
    if validation_metrics['accuracy'] > best_validation_accuracy:
        best_validation_accuracy = validation_metrics['accuracy']
        torch.save(checkpoint_payload(model, epoch, training_config, validation_metrics), CHECKPOINT_PATH)
        print(f'Saved best checkpoint: {CHECKPOINT_PATH}')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Run after training. The checkpoint is in /kaggle/working and can be downloaded from Output.
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
if checkpoint['class_names'] != list(ASL_CLASSES):
    raise ValueError('Checkpoint class ordering differs from src.config.ASL_CLASSES.')
best_model = create_model(pretrained=False).to(DEVICE)
best_model.load_state_dict(checkpoint['model_state_dict'])
test_metrics = evaluate_model(best_model, loaders['test'], DEVICE)

print(f"Test accuracy: {test_metrics['accuracy']:.4f}")
print(f"Macro precision: {sum(test_metrics['precision']) / len(ASL_CLASSES):.4f}")
print(f"Macro recall: {sum(test_metrics['recall']) / len(ASL_CLASSES):.4f}")
print(f"Macro F1: {test_metrics['macro_f1']:.4f}")

per_class = pd.DataFrame({
    'class': ASL_CLASSES, 'precision': test_metrics['precision'],
    'recall': test_metrics['recall'], 'f1': test_metrics['f1'],
})
display(per_class)

matrix = test_metrics['confusion_matrix']
plt.figure(figsize=(14, 12))
plt.imshow(matrix, cmap='Blues')
plt.xticks(range(len(ASL_CLASSES)), ASL_CLASSES, rotation=90)
plt.yticks(range(len(ASL_CLASSES)), ASL_CLASSES)
plt.xlabel('Predicted class')
plt.ylabel('True class')
plt.title('V1 test confusion matrix')
plt.colorbar()
plt.tight_layout()
plt.show()

print(f'Download this checkpoint from Kaggle Output: {CHECKPOINT_PATH}')
